# S4_03 — The Full RAG Flow with `VectorIndex`

> **Skilljar source**: Lesson **L05 — Implementing the RAG flow** (L04 "The full RAG flow" is theory-only, covered conceptually in Week_05.md §1.4)
> **Week_05.md mapping**: §1.4 "The Full RAG Flow" + §1.5 "Implementing the RAG Flow"
> **Original file**: `003_vectordb.ipynb`

## What You Will Learn

1. How to encapsulate embedding + storage + search into a single reusable `VectorIndex` class.
2. How cosine distance powers semantic retrieval — the core of RAG.
3. The **6-step RAG pipeline** (Week_05.md §1.4):
   1. Chunk source text
   2. Embed each chunk
   3. Store vectors in an index
   4. Embed the user query
   5. Find top-k nearest chunks
   6. Inject chunks into Claude's prompt

## ⚠️ Notebook is a Template with Placeholders

Cells 5, 6, 8, 9 are **intentionally left incomplete** in the Skilljar original — you fill them in while following the video. This is your hands-on exercise.

## Connection to Week_05.md

| Week_05.md Section | This Notebook |
|---|---|
| §1.4 "Step 1: Chunk Source Text" | Cell 5 (placeholder → use `chunk_by_section`) |
| §1.4 "Step 2–3: Embed + Store" | Cells 6–7 (`generate_embedding` + `store.add_documents`) |
| §1.4 "Step 4–5: Query + Search" | Cells 8–9 (generate query embedding + `store.search(..., k=2)`) |
| §1.5 "VectorIndex class" | Cell 3 (class definition) |
| §1.5 "Cosine distance formula" | Cell 3 `_cosine_distance` method |

## Setup · VoyageAI Client

Identical to S4_02. Loading `.env` and creating the client.

In [ ]:
# Client Setup
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

## Reuse · `chunk_by_section` from S4_01

Re-declared here for self-containment.

In [ ]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

## Reuse · `generate_embedding` from S4_02

**Note the subtle upgrade vs S4_02**: this version supports **both** a single string AND a list of strings (batch embedding). Batching is critical later to avoid VoyageAI rate-limiting when indexing many chunks at once.

> [!tip] Week_05.md §1.3
> In production you almost always want batched embedding — one HTTP round-trip per chunk is wasteful.

In [ ]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

## Core Component · `VectorIndex` Class

This is the centerpiece of S4_03. It combines three responsibilities in one object:

1. **Storage** — holds `(vector, document)` pairs
2. **Addition** — `add_document(doc)` embeds + stores; `add_vector(v, doc)` stores a precomputed vector
3. **Search** — `search(query, k)` returns the top-k closest documents by cosine distance

### Interface
```python
store = VectorIndex(embedding_fn=generate_embedding)
store.add_document({"content": "some text"})
store.search("user query", k=5)
```

> [!finding] Week_05.md §1.5 — Design choices
> - `distance_metric` is a parameter (`"cosine"` default) — swap to `"euclidean"` without changing user code.
> - `embedding_fn` is dependency-injected — you can mock it in tests, or plug a different provider.
> - `add_vector()` lets advanced users precompute vectors (e.g., with a GPU) and inject them.

### Cosine distance refresher (Week_05.md §1.5)
```
cosine_similarity = (a · b) / (||a|| · ||b||)      ∈ [-1, +1]
cosine_distance   = 1 - cosine_similarity          ∈ [0, 2]
```
Smaller distance ⇒ more similar. The `search` method sorts **ascending by distance**.

In [ ]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

## Step 1 · Load the Source Document

Plain file read — `report.md` is our gold-standard corporate annual report shared across all S4 notebooks.

In [ ]:
with open("./report.md", "r") as f:
    text = f.read()

## Step 2 · TODO — Chunk the Text by Section

**Your task**: call `chunk_by_section(text)` and save to `chunks`.

Skilljar originally leaves this as an empty cell — the Skilljar video shows the instructor filling it in live. Try it yourself:
```python
chunks = chunk_by_section(text)
print(f"Got {len(chunks)} chunks")   # expect ~13
```

In [ ]:
# 1. Chunk the text by section

## Step 3 · TODO — Generate Embeddings for Each Chunk

**Your task**: build a list of embeddings. The next cell uses `add_documents` so Skilljar's intended approach is the batch call:
```python
embeddings = generate_embedding(chunks)   # list in → list out (because S4_02's upgrade)
```

> [!tip] Why not a for-loop?
> Looping `generate_embedding(c) for c in chunks` works but triggers VoyageAI rate limits on the free tier. Always batch when you can.

In [ ]:
# 2. Generate embeddings for each chunk

## Step 4 · Create a Vector Store and Bulk-Load

> ⚠️ **API gap exposed here**: the original `VectorIndex` class in cell 3 has only `add_document` (singular). The Skilljar original adds `add_documents` (plural) either as a homework exercise or by silently updating the class behind the scenes.
>
> **If this cell errors** with `AttributeError: 'VectorIndex' object has no attribute 'add_documents'`, add this method to the class in cell 3:
> ```python
> def add_documents(self, docs):
>     contents = [d["content"] for d in docs]
>     vectors = self._embedding_fn(contents)
>     for v, d in zip(vectors, docs):
>         self.add_vector(v, d)
> ```
>
> The fully-upgraded `VectorIndex` appears in **S4_05**, so you can also copy from there.

In [ ]:
# 3. Create a vector store and add each embedding to it
# Note: converted to a bulk operation to avoid rate limiting errors from VoyageAI
store = VectorIndex()
store.add_documents([{"content": chunk} for chunk in chunks])

## Step 5 · TODO — Embed the User Query

**Your task**: write a query and embed it. Example (Week_05.md §1.5 demo query):
```python
user_query = "What did the software engineering department do last year?"
query_embedding = generate_embedding(user_query)
```

In [ ]:
# 4. Some time later, a user will ask a question. Generate an embedding for it

## Step 6 · TODO — Retrieve the Top-k Most Relevant Chunks

**Your task**: call `store.search(query_embedding, k=2)` and inspect the results.

> [!finding] Week_05.md §1.5 expected output
> For the query *"What did the software engineering department do last year?"* the top-2 hits should be **Section 2 (Software Engineering)** — distances ≈ 0.71 and 0.72. The exact values depend on the VoyageAI model version, but the ranking should be stable.

> [!action] Extension (optional)
> After retrieval, feed the top-2 chunks into Claude as grounded context:
> ```python
> import anthropic
> client = anthropic.Anthropic()
> retrieved_text = "\n---\n".join(d["content"] for d, _ in results)
> response = client.messages.create(
>     model="claude-haiku-4-5",
>     max_tokens=1024,
>     system="Answer based only on the provided context.",
>     messages=[{"role": "user", "content": f"Context:\n{retrieved_text}\n\nQuestion: {user_query}"}],
> )
> print(response.content[0].text)
> ```
> This closes the loop on the 6-step RAG flow.

In [ ]:
# 5. Search the store with the embedding, find the 2 most relevant chunks

## Wrap-up

You have built a working semantic search engine grounded to Claude. The next gap we close in **S4_04** is the *semantic-only blind spot* — queries containing rare literal tokens (e.g., `INC-2023-Q4-011`) that embeddings underweight.

> [!ref] Skilljar L04 (theory) + L05 (code)
> Week_05.md §1.4 "The Full RAG Flow" · §1.5 "Implementing the RAG Flow"